# 22. Instance Segmentation 개념

이 노트북은 `21_U-Net_간단_실습.ipynb` 다음 단계로, semantic segmentation과 instance segmentation의 차이를 정리합니다.

Semantic segmentation은 모든 픽셀의 클래스를 예측합니다. 하지만 같은 클래스의 객체가 여러 개 있을 때, 각각을 따로 구분하지는 않습니다. Instance segmentation은 여기서 한 단계 더 나아가 **같은 클래스 안에서도 개별 객체 instance를 구분**합니다.

이번 노트북의 목표는 다음과 같습니다.

- semantic segmentation과 instance segmentation의 차이를 이해합니다.
- class map과 instance map의 차이를 구분합니다.
- instance segmentation이 detection과 segmentation을 함께 다루는 문제임을 이해합니다.
- Mask R-CNN으로 넘어가기 위한 기본 관점을 준비합니다.


## 22-1. 준비

간단한 배열을 직접 만들어 같은 장면을 semantic map과 instance map으로 각각 표현해 봅니다.


In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False


## 22-2. Semantic map: 클래스만 구분한다

Semantic segmentation의 출력은 픽셀마다 class id를 가집니다. 예를 들어 자동차가 두 대 있어도 둘 다 `car` 클래스입니다.

아래 class map에서는 두 개의 자동차 영역이 모두 같은 값 `2`를 가집니다.


In [ ]:
semantic_map = np.zeros((12, 18), dtype=np.int64)
semantic_map[8:, :] = 1          # road
semantic_map[5:8, 2:7] = 2       # car
semantic_map[6:9, 11:16] = 2     # car
semantic_map[3:8, 8:10] = 3      # person

semantic_names = {0: 'background', 1: 'road', 2: 'car', 3: 'person'}
semantic_colors = ['#d9d9d9', '#4c78a8', '#f58518', '#54a24b']

plt.figure(figsize=(8, 4))
plt.imshow(semantic_map, cmap=ListedColormap(semantic_colors), vmin=0, vmax=3)
plt.title('Semantic map: 두 자동차가 모두 car 클래스')
plt.xticks([])
plt.yticks([])
plt.show()


## 22-3. Instance map: 개별 객체를 구분한다

Instance segmentation에서는 같은 클래스의 객체라도 서로 다른 instance id를 가질 수 있습니다.

예를 들어 두 자동차는 모두 class가 `car`이지만, instance 관점에서는 `car #1`, `car #2`로 분리됩니다.


In [ ]:
instance_map = np.zeros((12, 18), dtype=np.int64)
instance_map[5:8, 2:7] = 1       # car instance 1
instance_map[6:9, 11:16] = 2     # car instance 2
instance_map[3:8, 8:10] = 3      # person instance 1

instance_colors = ['#d9d9d9', '#f58518', '#e45756', '#54a24b']

plt.figure(figsize=(8, 4))
plt.imshow(instance_map, cmap=ListedColormap(instance_colors), vmin=0, vmax=3)
plt.title('Instance map: 같은 car 클래스도 객체별로 분리')
plt.xticks([])
plt.yticks([])
plt.show()

print('instance id 1: car #1')
print('instance id 2: car #2')
print('instance id 3: person #1')


## 22-4. Semantic과 instance의 차이

두 문제의 차이는 질문 자체가 다릅니다.

- Semantic segmentation: `이 픽셀은 어떤 클래스인가?`
- Instance segmentation: `이 픽셀은 어떤 클래스이며, 어느 객체에 속하는가?`

따라서 instance segmentation은 클래스 예측뿐 아니라 개별 객체를 분리하는 과정이 필요합니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(semantic_map, cmap=ListedColormap(semantic_colors), vmin=0, vmax=3)
axes[0].set_title('Semantic segmentation')

axes[1].imshow(instance_map, cmap=ListedColormap(instance_colors), vmin=0, vmax=3)
axes[1].set_title('Instance segmentation')

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()


## 22-5. Instance segmentation은 detection과 segmentation의 결합이다

Instance segmentation은 보통 다음 정보를 함께 예측합니다.

- 객체 class
- 객체 bounding box
- 객체별 mask
- confidence score

이 때문에 instance segmentation 모델은 detection 모델의 아이디어를 많이 사용합니다. 대표적인 모델인 Mask R-CNN도 Faster R-CNN의 detection 구조 위에 mask branch를 추가한 방식입니다.


In [ ]:
instance_prediction = [
    {'class': 'car', 'score': 0.96, 'bbox_xyxy': [2, 5, 7, 8], 'mask_id': 1},
    {'class': 'car', 'score': 0.91, 'bbox_xyxy': [11, 6, 16, 9], 'mask_id': 2},
    {'class': 'person', 'score': 0.88, 'bbox_xyxy': [8, 3, 10, 8], 'mask_id': 3},
]

for pred in instance_prediction:
    print(pred)


## 22-6. Panoptic segmentation은 무엇인가?

Segmentation 문제를 공부하다 보면 panoptic segmentation이라는 용어도 나옵니다. Panoptic segmentation은 semantic segmentation과 instance segmentation을 통합하려는 문제입니다.

- stuff: 도로, 하늘, 벽처럼 셀 수 없는 배경성 영역
- thing: 사람, 자동차, 자전거처럼 개별 객체로 셀 수 있는 대상

Panoptic segmentation은 stuff는 semantic하게 나누고, thing은 instance 단위로 나눕니다. 지금 단계에서는 `semantic과 instance를 합친 더 포괄적인 문제` 정도로 이해하면 충분합니다.


## 22-7. 언제 어떤 segmentation을 사용할까?

- Semantic segmentation: 도로, 차선, 하늘, 건물처럼 클래스별 영역이 중요할 때
- Instance segmentation: 사람 수, 차량 수, 개별 물체의 mask가 중요할 때
- Panoptic segmentation: 배경 영역과 개별 객체를 모두 통합적으로 이해해야 할 때

문제의 목표가 `클래스 영역`인지, `개별 객체`인지에 따라 필요한 segmentation 방식이 달라집니다.


## 22-8. 정리

- Semantic segmentation은 픽셀마다 클래스만 예측합니다.
- Instance segmentation은 같은 클래스 안에서도 개별 객체를 구분합니다.
- Instance segmentation 결과는 class, box, score, mask를 함께 포함하는 경우가 많습니다.
- Mask R-CNN은 detection 모델에 mask 예측 branch를 추가한 대표적인 instance segmentation 모델입니다.
- 다음 노트북에서는 Mask R-CNN의 핵심 아이디어와 RoI Align의 역할을 살펴봅니다.
